In [0]:
pip freeze

In [0]:
!pip install scikit-learn==1.4.2 optbinning==0.19.0

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from optbinning import OptimalBinning
from sklearn.preprocessing import OneHotEncoder
from scipy import stats as stat
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import warnings
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy import stats as stat
from scipy.stats import ks_2samp


%matplotlib inline
pd.options.display.max_columns = None
pd.options.display.max_rows = None
warnings.filterwarnings('ignore')

In [0]:
df_og = spark.table("bigquery_credit_analytics_catalog.credit_analytics.loan_data")
df = df_og
display(df)

## General Pre-Processing of the Data

In [0]:
# tratando a variável de emp_length
df = df.withColumn('emp_length_int',
                F.regexp_replace(F.col('emp_length'), r'\+ years', ''))
df = df.withColumn('emp_length_int',
                F.regexp_replace(F.col('emp_length_int'), '< 1 year', '0'))
df = df.withColumn('emp_length_int',
                F.regexp_replace(F.col('emp_length_int'), 'n/a', '0'))
df = df.withColumn('emp_length_int',
                F.regexp_replace(F.col('emp_length_int'), ' years', ''))
df = df.withColumn('emp_length_int',
                F.regexp_replace(F.col('emp_length_int'), ' year', ''))
df = df.withColumn('emp_length_int',
                F.trim(F.col('emp_length_int')).cast('int'))

# tratando a variável de term
df = df.withColumn('term_int',
                F.trim(F.col('term')).substr(1, 2).cast('int'))

# tratando a variável de earliest_cr_line
df = df.withColumn('earliest_cr_line_date',
                F.to_date(F.col('earliest_cr_line'), 'MMM-yyyy'))

# tratando a variável de issue_d
df = df.withColumn('issue_d_date',
                F.to_date(F.col('issue_d'), 'MMM-yyyy'))

# criando a variável de número de meses para a earliest_cr_line
df = df.withColumn('mths_since_earliest_cr_line',
                F.round(F.months_between(F.lit('2017-12-01').cast('date'), F.col('earliest_cr_line_date'))).cast('int'))

# criando a variável de número de meses para a issue_d
df = df.withColumn('mths_since_issue_d',
                F.round(F.months_between(F.lit('2017-12-01').cast('date'), F.col('issue_d_date'))).cast('int'))

default_list = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)']
df = df.withColumn('default_bin',
                F.when(F.col('loan_status').isin(default_list), 1).otherwise(0))

In [0]:
df.select(*['default_bin']).describe().show()

In [0]:
display(df.select([F.sum(F.isnull(c).cast("int")).alias(c) for c in df.columns]))

## Treating Missing Data

In [0]:
df = df.withColumn('total_rev_hi_lim',
                F.when(F.col('total_rev_hi_lim').isNull(), F.col('funded_amnt')).otherwise(F.col('total_rev_hi_lim')))

treat_with_zeros = ['mths_since_earliest_cr_line',
                    'acc_now_delinq',
                    'total_acc',
                    'pub_rec',
                    'open_acc',
                    'inq_last_6mths',
                    'delinq_2yrs',
                    'emp_length_int']

for c in treat_with_zeros:

    df = df.withColumn(c,
                F.when(F.col(c).isNull(), 0).otherwise(F.col(c)))

## Separating the data: Training, Validation and Testing

In [0]:
df_test = df.filter(F.year(F.col('issue_d_date')) == 2015)

df_train_validation = df.filter(F.year(F.col('issue_d_date')) < 2015)

df_train = df_train_validation.filter(F.col('issue_d_date') < F.to_date(F.lit('2014-09-01')))
df_validation = df_train_validation.filter(F.col('issue_d_date') >= F.to_date(F.lit('2014-09-01')))

In [0]:
df_validation.select(*['default_bin']).describe().show()

In [0]:
median_annual_inc = df_train.filter(F.col('annual_inc').isNotNull()).agg(F.median('annual_inc')).collect()[0][0]

df_train = df_train.withColumn('annual_inc',
                F.when(F.col('annual_inc').isNull(), median_annual_inc).otherwise(F.col('annual_inc')))

df_validation = df_validation.withColumn('annual_inc',
                F.when(F.col('annual_inc').isNull(), median_annual_inc).otherwise(F.col('annual_inc')))

df_test = df_test.withColumn('annual_inc',
                F.when(F.col('annual_inc').isNull(), median_annual_inc).otherwise(F.col('annual_inc')))

print(median_annual_inc)

In [0]:
display(df_train)

### Performing Feature Binning

In [0]:
TARGET = 'default_bin'  # binary target column (0/1)

NUMERICAL_FEATURES = [
    'loan_amnt', 'funded_amnt', 'dti', 'annual_inc', 'installment',
    'int_rate', 'total_rev_hi_lim', 'mths_since_last_record',
    'mths_since_last_delinq', 'total_acc', 'acc_now_delinq', 'pub_rec',
    'open_acc', 'inq_last_6mths', 'delinq_2yrs', 'emp_length_int',
    'term_int', 'mths_since_earliest_cr_line'
]

CATEGORICAL_FEATURES = [
    'purpose', 'initial_list_status', 'verification_status',
    'addr_state', 'home_ownership', 'grade', 'sub_grade'
]

# Features where NaN should be treated as a separate bin
NULL_AS_BIN_FEATURES = [
    'mths_since_last_record',
    'mths_since_last_delinq',
]

MAX_BINS = 15

In [0]:
df_train_pd = df_train.toPandas()

int32_cols = df_train_pd.select_dtypes(include='int32').columns.tolist()
df_train_pd[int32_cols] = df_train_pd[int32_cols].astype('int64')

In [0]:
def run_binning(df: pd.DataFrame,
                target: str,
                numerical_features: list,
                categorical_features: list,
                null_as_bin_features: list,
                max_bins: int = 15):
    """
    Parameters
    ----------
    df                   : Input DataFrame
    target               : Binary target column name (0/1)
    numerical_features   : List of numerical feature names
    categorical_features : List of categorical feature names
    null_as_bin_features : Features where NaN is treated as a separate bin
    max_bins             : Maximum number of bins per feature

    Returns
    -------
    df_binned    : DataFrame with binned features (original columns replaced with WoE values)
    summary_df   : Table with bin details and IV per feature
    """

    y = df[target]
    df_binned = df.copy()
    summary_records = []

    all_features = (
        [(f, 'numerical')   for f in numerical_features] +
        [(f, 'categorical') for f in categorical_features]
    )

    for feature, dtype in all_features:
        if feature not in df.columns:
            print(f"[SKIP] '{feature}' not found in DataFrame.")
            continue

        x = df[feature]
        special_codes = [np.nan] if feature in null_as_bin_features else []

        try:
            
            n_categories = df[feature].nunique() if dtype == 'categorical' else None

            binner = OptimalBinning(
                name=feature,
                dtype=dtype,
                max_n_bins=max_bins,
                max_n_prebins=max(512, n_categories * 2) if dtype == 'categorical' else 20,
                special_codes=special_codes if dtype == 'numerical' else [],
                solver='cp' if dtype == 'numerical' else 'mip',
                monotonic_trend='auto_asc_desc'
            )

            binner.fit(x.values, y.values)
            binning_table = binner.binning_table.build()
            iv_total = binning_table['IV'].iloc[-1]

            n_rows = len(binning_table)

            for i, (_, row) in enumerate(binning_table.iterrows()):
    
                # Skip totals row (always last)
                if i == n_rows - 1:
                    continue

                # Check if bin is a string (Special/Missing) or a list (actual bin)
                if isinstance(row['Bin'], str):
                    bin_label = row['Bin']  # 'Special' or 'Missing'
                else:
                    bin_label = str(list(row['Bin']))  # convert list like ['A', 'B'] to string

                summary_records.append({
                    'Feature':    feature,
                    'Type':       dtype,
                    'Bin':        bin_label,
                    'Count':      row['Count'],
                    'Count (%)':  round(row['Count (%)'], 4),
                    'Events':     row['Event'],
                    'Event Rate': round(row['Event rate'], 4),
                    'WoE':        round(row['WoE'], 4),
                    'IV (bin)':   round(row['IV'], 4),
                    'IV (total)': round(iv_total, 4),
                })

            # ── Transform column to WoE ──────────────────────────────────
            df_binned[feature] = binner.transform(x.values, metric='bins')

        except Exception as e:
            print(f"[ERROR] Could not bin '{feature}': {e}")
            
            continue

    summary_df = pd.DataFrame(summary_records)

    return df_binned, summary_df



def show_iv_summary(summary_df: pd.DataFrame):
    """Returns a concise IV summary per feature, sorted by IV descending."""
    iv_summary = (
        summary_df[summary_df['Bin'] != 'Totals']
        .groupby(['Feature', 'Type'])['IV (total)']
        .first()
        .reset_index()
        .sort_values('IV (total)', ascending=False)
        .reset_index(drop=True)
    )
    return iv_summary

In [0]:
df_binned, summary_df = run_binning(
        df=df_train_pd,
        target=TARGET,
        numerical_features=NUMERICAL_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        null_as_bin_features=NULL_AS_BIN_FEATURES,
        max_bins=MAX_BINS,
    )

In [0]:
iv_summary = show_iv_summary(summary_df)

In [0]:
df_binned.head(15)

In [0]:
summary_df

In [0]:
iv_summary

## Selecting Features:
### Removing Features with low IV 

In [0]:
features_selected_iv = list(iv_summary[iv_summary['IV (total)'] > 0.01]['Feature'])
features_selected_iv


## Selecting Features:
### Selecting features based on statistical significance and KS (Kolmogorov-Smirnov) 

In [0]:
ref_df = summary_df.loc[(summary_df['Feature'].isin(features_selected_iv)) & ~(summary_df['Bin'].isin(['Special', 'Missing']))][['Feature', 'Bin', 'WoE']]

reference_cats_df = ref_df.loc[ref_df.groupby('Feature')['WoE'].idxmin()][['Feature', 'Bin']]

reference_dict = dict(zip(reference_cats_df['Feature'], reference_cats_df['Bin']))

reference_dict

In [0]:
def apply_manual_bins(df_in):

    df = df_in.copy()

    # ─────────────────────────────────────────────
    # NUMERICAL FEATURES
    # ─────────────────────────────────────────────

    # dti
    df['dti'] = pd.cut(df['dti'],
        bins=[-np.inf, 8.89, 10.30, 12.14, 13.38, 14.53, 15.59, 17.13, 18.80, 20.18, 21.44, 25.85, 29.01, np.inf],
        labels=['(-inf, 8.89)', '[8.89, 10.30)', '[10.30, 12.14)', '[12.14, 13.38)',
                '[13.38, 14.53)', '[14.53, 15.59)', '[15.59, 17.13)', '[17.13, 18.80)',
                '[18.80, 20.18)', '[20.18, 21.44)', '[21.44, 25.85)', '[25.85, 29.01)', '[29.01, inf)'])

    # annual_inc
    df['annual_inc'] = pd.cut(df['annual_inc'],
        bins=[-np.inf, 28338.00, 37086.20, 40371.00, 49464.10, 60996.50,
              66098.00, 70703.40, 80046.22, 90230.50, 100131.00, 120012.00, np.inf],
        labels=['(-inf, 28338.00)', '[28338.00, 37086.20)', '[37086.20, 40371.00)',
                '[40371.00, 49464.10)', '[49464.10, 60996.50)', '[60996.50, 66098.00)',
                '[66098.00, 70703.40)', '[70703.40, 80046.22)', '[80046.22, 90230.50)',
                '[90230.50, 100131.00)', '[100131.00, 120012.00)', '[120012.00, inf)'])

    # int_rate
    df['int_rate'] = pd.cut(df['int_rate'],
        bins=[-np.inf, 7.74, 8.92, 10.15, 11.01, 12.01, 13.05, 13.98, 15.12, 15.61, 17.57, 19.01, 20.99, np.inf],
        labels=['(-inf, 7.74)', '[7.74, 8.92)', '[8.92, 10.15)', '[10.15, 11.01)',
                '[11.01, 12.01)', '[12.01, 13.05)', '[13.05, 13.98)', '[13.98, 15.12)',
                '[15.12, 15.61)', '[15.61, 17.57)', '[17.57, 19.01)', '[19.01, 20.99)', '[20.99, inf)'])

    # total_rev_hi_lim
    df['total_rev_hi_lim'] = pd.cut(df['total_rev_hi_lim'],
        bins=[-np.inf, 5940.50, 11726.00, 20294.50, 28118.50, 36034.50, 44662.00, 55862.00, np.inf],
        labels=['(-inf, 5940.50)', '[5940.50, 11726.00)', '[11726.00, 20294.50)',
                '[20294.50, 28118.50)', '[28118.50, 36034.50)', '[36034.50, 44662.00)',
                '[44662.00, 55862.00)', '[55862.00, inf)'])

    # inq_last_6mths
    df['inq_last_6mths'] = pd.cut(df['inq_last_6mths'],
        bins=[-np.inf, 0.50, 1.50, 2.50, np.inf],
        labels=['(-inf, 0.50)', '[0.50, 1.50)', '[1.50, 2.50)', '[2.50, inf)'])

    # term_int
    df['term_int'] = pd.cut(df['term_int'],
        bins=[-np.inf, 3.00, np.inf],
        labels=['(-inf, 48.00)', '[48.00, inf)'])

    # mths_since_earliest_cr_line
    df['mths_since_earliest_cr_line'] = pd.cut(df['mths_since_earliest_cr_line'],
        bins=[-np.inf, 145.50, 168.50, 204.50, 228.50, 247.50, 266.50, 287.50, 353.50, np.inf],
        labels=['(-inf, 145.50)', '[145.50, 168.50)', '[168.50, 204.50)', '[204.50, 228.50)',
                '[228.50, 247.50)', '[247.50, 266.50)', '[266.50, 287.50)', '[287.50, 353.50)', '[353.50, inf)'])

    # ─────────────────────────────────────────────
    # CATEGORICAL FEATURES
    # ─────────────────────────────────────────────

    # purpose
    purpose_map = {
        'credit_card':        "['credit_card', 'car']",
        'car':                "['credit_card', 'car']",
        'major_purchase':     "['major_purchase', 'home_improvement']",
        'home_improvement':   "['major_purchase', 'home_improvement']",
        'wedding':            "['wedding', 'debt_consolidation', 'vacation']",
        'debt_consolidation': "['wedding', 'debt_consolidation', 'vacation']",
        'vacation':           "['wedding', 'debt_consolidation', 'vacation']",
        'medical':            "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'house':              "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'other':              "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'moving':             "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'renewable_energy':   "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'small_business':     "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'educational':     "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']"
    }
    df['purpose'] = df['purpose'].map(purpose_map).fillna("['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']") # fill NaN with the worst case scenario

    # initial_list_status
    df['initial_list_status'] = df['initial_list_status'].map({
        'w': "['w']",
        'f': "['f']",
    }).fillna("['f']") # fill NaN with the worst case scenario

    # verification_status
    df['verification_status'] = df['verification_status'].map({
        'Not Verified':    "['Not Verified']",
        'Source Verified': "['Source Verified']",
        'Verified':        "['Verified']",
    }).fillna("['Verified']") # fill NaN with the worst case scenario

    # addr_state
    addr_state_map = {
        # Bin 1
        'ME': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'DC': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'WY': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'ID': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'NH': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'WV': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'AK': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'CO': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'KS': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'MS': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        # Bin 2
        'MT': "['MT', 'VT', 'SC', 'TX']",
        'VT': "['MT', 'VT', 'SC', 'TX']",
        'SC': "['MT', 'VT', 'SC', 'TX']",
        'TX': "['MT', 'VT', 'SC', 'TX']",
        # Bin 3
        'CT': "['CT', 'IL']",
        'IL': "['CT', 'IL']",
        # Bin 4
        'OR': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'WI': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'WA': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'MN': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'GA': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'SD': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        # Bin 5
        'DE': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'MA': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'IN': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'KY': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'RI': "['DE', 'MA', 'IN', 'KY', 'RI']",
        # Bin 6
        'OH': "['OH', 'PA', 'AZ', 'LA']",
        'PA': "['OH', 'PA', 'AZ', 'LA']",
        'AZ': "['OH', 'PA', 'AZ', 'LA']",
        'LA': "['OH', 'PA', 'AZ', 'LA']",
        # Bin 7
        'UT': "['UT', 'MI', 'VA']",
        'MI': "['UT', 'MI', 'VA']",
        'VA': "['UT', 'MI', 'VA']",
        # Bin 8
        'CA': "['CA', 'TN', 'AR']",
        'TN': "['CA', 'TN', 'AR']",
        'AR': "['CA', 'TN', 'AR']",
        # Bin 9
        'NC': "['NC', 'OK', 'MD']",
        'OK': "['NC', 'OK', 'MD']",
        'MD': "['NC', 'OK', 'MD']",
        # Bin 10
        'MO': "['MO', 'NY', 'NJ', 'NM']",
        'NY': "['MO', 'NY', 'NJ', 'NM']",
        'NJ': "['MO', 'NY', 'NJ', 'NM']",
        'NM': "['MO', 'NY', 'NJ', 'NM']",
        # Bin 11
        'AL': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'HI': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'FL': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'NV': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'IA': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'NE': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
    }
    df['addr_state'] = df['addr_state'].map(addr_state_map).fillna("['AL', 'HI', 'FL', 'NV', 'IA', 'NE']") # fill NaN with the worst case scenario

    # home_ownership
    df['home_ownership'] = df['home_ownership'].map({
        'MORTGAGE': "['MORTGAGE']",
        'OWN':      "['OWN']",
        'RENT':     "['RENT', 'NONE', 'OTHER']",
        'NONE':     "['RENT', 'NONE', 'OTHER']",
        'OTHER':    "['RENT', 'NONE', 'OTHER']",
    }).fillna("['RENT', 'NONE', 'OTHER']") # fill NaN with the worst case scenario

    # grade
    df['grade'] = df['grade'].map({
        'A': "['A']",
        'B': "['B']",
        'C': "['C']",
        'D': "['D']",
        'E': "['E', 'F', 'G']",
        'F': "['E', 'F', 'G']",
        'G': "['E', 'F', 'G']",
    }).fillna("['E', 'F', 'G']") # fill NaN with the worst case scenario

    # sub_grade
    last_bin = "['E2', 'E3', 'E4', 'F1', 'E5', 'F2', 'G4', 'F3', 'F4', 'G2', 'G5', 'G3', 'F5', 'G1']"
    sub_grade_map = {
        'A1': "['A1', 'A2', 'A3']", 'A2': "['A1', 'A2', 'A3']", 'A3': "['A1', 'A2', 'A3']",
        'A4': "['A4', 'A5']",       'A5': "['A4', 'A5']",
        'B1': "['B1']",
        'B2': "['B2']",
        'B3': "['B3']",
        'B4': "['B4']",
        'B5': "['B5']",
        'C1': "['C1']",
        'C2': "['C2']",
        'C3': "['C3']",
        'C4': "['C4']",
        'C5': "['C5', 'D1']",       'D1': "['C5', 'D1']",
        'D2': "['D2', 'D3']",       'D3': "['D2', 'D3']",
        'D4': "['D4', 'D5', 'E1']", 'D5': "['D4', 'D5', 'E1']", 'E1': "['D4', 'D5', 'E1']",
        'E2': last_bin, 'E3': last_bin, 'E4': last_bin, 'E5': last_bin,
        'F1': last_bin, 'F2': last_bin, 'F3': last_bin, 'F4': last_bin, 'F5': last_bin,
        'G1': last_bin, 'G2': last_bin, 'G3': last_bin, 'G4': last_bin, 'G5': last_bin,
    }
    df['sub_grade'] = df['sub_grade'].map(sub_grade_map).fillna("['E2', 'E3', 'E4', 'F1', 'E5', 'F2', 'G4', 'F3', 'F4', 'G2', 'G5', 'G3', 'F5', 'G1']") # fill NaN with the worst case scenario

    return df

In [0]:

# ── 1. Custom LR with p-values ────────────────────────────────────────────────

class LogisticRegression_with_p_values:
    def __init__(self, max_iter=1000):
        # C=1e9 ≈ no regularization
        self.model = LogisticRegression(C=1e9, max_iter=max_iter, solver='lbfgs')

    def fit(self, X, y):
        self.model.fit(X, y)
        denom = 2.0 * (1.0 + np.cosh(self.model.decision_function(X)))
        denom = np.tile(denom, (X.shape[1], 1)).T
        F_ij = np.dot((X / denom).T, X)
        Cramer_Rao = np.linalg.inv(F_ij)
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores]
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


# ── 2. KS statistic ───────────────────────────────────────────────────────────

def compute_ks(y_true, y_prob):
    """KS between score distributions of events and non-events."""
    scores_event    = y_prob[y_true == 1]
    scores_nonevent = y_prob[y_true == 0]
    ks_stat, _      = ks_2samp(scores_event, scores_nonevent)
    return ks_stat


# ── 3. P-value feature filter ─────────────────────────────────────────────────

def feature_is_significant(p_values_for_feature: list,
                            alpha: float = 0.05,
                            many_categories_threshold: int = 10) -> bool:
    """
    Rules
    -----
    • All categories insignificant                              → exclude
    • All significant OR at least one significant               → keep
      EXCEPT: 10+ categories and only exactly 1 significant    → exclude
    """
    n_cats = len(p_values_for_feature)
    n_sig  = sum(p < alpha for p in p_values_for_feature)

    if n_sig == 0:
        return False
    if n_cats >= many_categories_threshold and n_sig == 1:
        return False
    return True


def encode_features(df: pd.DataFrame,
                    features: list,
                    reference_dict: dict) -> tuple:

    # Step 1: fit without drop to learn actual category strings in the data
    enc_temp = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    enc_temp.fit(df[features].astype(str))

    # Step 2: validate reference exists in actual data categories
    drop_cats = []
    for feat, cats in zip(features, enc_temp.categories_):
        ref = reference_dict[feat]
        if ref not in cats:
            raise ValueError(
                f"\nReference mismatch for '{feat}'."
                f"\n  Expected : '{ref}'"
                f"\n  Available: {list(cats)}"
            )
        drop_cats.append(ref)  # plain list, not np.array

    # Step 3: fit final encoder with validated plain list
    enc = OneHotEncoder(
        drop=drop_cats,          # ← plain list
        sparse_output=False,
        handle_unknown='ignore'
    )
    X = enc.fit_transform(df[features].astype(str))
    return X.astype(float), enc

# ── 5. Map flat p-values back to original features ───────────────────────────

def map_pvalues_to_features(enc: OneHotEncoder,
                             features: list,
                             p_values: list) -> dict:
    """
    Returns {feature_name: [p_val_for_non_reference_cat, ...]}

    Because the reference category differs per feature, we use
    enc.drop_idx_ to know which category was actually dropped,
    and reconstruct the kept categories in the correct order.
    """
    feature_pvalues = {}
    idx = 0
    for i, feat in enumerate(features):
        all_cats  = enc.categories_[i]
        drop_idx  = int(enc.drop_idx_[i])
        kept_cats = [c for j, c in enumerate(all_cats) if j != drop_idx]
        n_dummies = len(kept_cats)
        feature_pvalues[feat] = {
            cat: p_values[idx + k]
            for k, cat in enumerate(kept_cats)
        }
        idx += n_dummies
    return feature_pvalues


# ── 6. Fit & evaluate one candidate subset ───────────────────────────────────

def evaluate_subset(df: pd.DataFrame,
                    y: pd.Series,
                    df_val,
                    y_val,
                    features: list,
                    reference_dict: dict,
                    alpha: float = 0.05,
                    many_cats_threshold: int = 10,
                    max_iter: int = 1000) -> tuple:
    """
    Fit the model on `features`, compute KS, and flag each feature
    as significant / insignificant according to the p-value rules.

    Returns
    -------
    ks            : float
    all_sig       : bool   – True if every feature passes the p-value rule
    feature_flags : dict   – {feature: True/False}
    feat_pvals    : dict   – {feature: {category: p_value}}
    model         : fitted LogisticRegression_with_p_values
    enc           : fitted OneHotEncoder
    """
    X, enc = encode_features(df, features, reference_dict)
    X_val  = enc.transform(df_val[features].astype(str)).astype(float)

    model = LogisticRegression_with_p_values(max_iter=max_iter)
    model.fit(X, y.values)

    y_prob = model.predict_proba(X_val)[:, 1]
    ks     = compute_ks(y_val.values, y_prob)

    feat_pvals    = map_pvalues_to_features(enc, features, model.p_values)
    feature_flags = {
        feat: feature_is_significant(
            list(cat_pvals.values()), alpha, many_cats_threshold
        )
        for feat, cat_pvals in feat_pvals.items()
    }
    all_sig = all(feature_flags.values())

    return ks, all_sig, feature_flags, feat_pvals, model, enc


# ── 7. Main selection routine ─────────────────────────────────────────────────

def select_features(df: pd.DataFrame,
                    y: pd.Series,
                    df_val,
                    y_val,
                    all_features: list,
                    reference_dict: dict,
                    alpha: float = 0.05,
                    many_cats_threshold: int = 10,
                    max_iter: int = 1000,
                    verbose: bool = True) -> tuple:
    """
    Iterative p-value pruning followed by a KS comparison.

    Strategy
    --------
    1. Fit on all features; drop features that fail the significance rule.
    2. Refit on survivors; repeat until all remaining features are significant.
    3. Report KS of the pruned model vs. the full model.
       Return the pruned model (significance constraint is always respected).

    Returns
    -------
    best_features : list
    results       : dict  (ks, feature_flags, feat_pvals, model, enc)
    """
    current_features = list(all_features)

    for iteration in range(len(all_features)):
        ks, all_sig, flags, feat_pvals, model, enc = evaluate_subset(
            df, y, df_val, y_val, current_features, reference_dict,
            alpha, many_cats_threshold, max_iter
        )

        if verbose:
            print(f"\n── Iteration {iteration + 1} " + "─" * 50)
            print(f"   Features ({len(current_features)}): {current_features}")
            print(f"   KS: {ks:.4f}")
            print(f"   Significance flags: {flags}")

        if all_sig:
            if verbose:
                print("   ✔ All features significant – pruning complete.")
            break

        failing = [f for f, ok in flags.items() if not ok]
        if verbose:
            print(f"   ✘ Dropping: {failing}")
        current_features = [f for f in current_features if f not in failing]

        if not current_features:
            raise ValueError("No features survived the significance filter.")

    pruned_features = current_features
    ks_pruned       = ks

    # Compare against the full model for reference
    ks_full, _, flags_full, _, _, _ = evaluate_subset(
        df, y, df_val, y_val, all_features, reference_dict,
        alpha, many_cats_threshold, max_iter
    )

    if verbose:
        print(f"\n── Final Summary " + "─" * 50)
        print(f"   Full model   – {len(all_features):>2} features | KS: {ks_full:.4f}")
        print(f"   Pruned model – {len(pruned_features):>2} features | KS: {ks_pruned:.4f}")
        delta = ks_pruned - ks_full
        print(f"   KS delta (pruned − full): {delta:+.4f}")
        print(f"   ✔ Returning pruned model (all features significant).")

    return pruned_features, dict(
        ks=ks_pruned,
        feature_flags=flags,
        feat_pvals=feat_pvals,
        model=model,
        enc=enc
    )


# ── 8. Reporting helper ───────────────────────────────────────────────────────

def print_model_report(features: list,
                       results: dict,
                       reference_dict: dict,
                       alpha: float = 0.05) -> pd.DataFrame:
    """
    Print a clean table: feature | category | reference | coef | p-value | significant.
    Returns the DataFrame as well for further use.
    """
    model     = results['model']
    enc       = results['enc']
    feat_pvals = results['feat_pvals']

    rows = []
    coef_iter = iter(model.coef_[0])

    for i, feat in enumerate(features):
        all_cats = enc.categories_[i]
        drop_idx = int(enc.drop_idx_[i])
        ref_cat  = all_cats[drop_idx]

        for j, cat in enumerate(all_cats):
            if j == drop_idx:
                # Reference category row — no coefficient or p-value
                rows.append(dict(
                    feature=feat,
                    category=cat,
                    is_reference=True,
                    coefficient=np.nan,
                    p_value=np.nan,
                    significant=np.nan
                ))
            else:
                coef  = next(coef_iter)
                p_val = feat_pvals[feat][cat]
                rows.append(dict(
                    feature=feat,
                    category=cat,
                    is_reference=False,
                    coefficient=coef,
                    p_value=p_val,
                    significant=p_val < alpha
                ))

    report = pd.DataFrame(rows)

    print(f"\nModel Report  |  KS = {results['ks']:.4f}\n")
    print(report.to_string(index=False))
    return report

In [0]:
df_validation_pd = df_validation.toPandas()

df_validation_eng = df_validation_pd[['int_rate',
                            'sub_grade',
                            'grade',
                            'annual_inc',
                            'term_int',
                            'inq_last_6mths',
                            'total_rev_hi_lim',
                            'purpose',
                            'dti',
                            'home_ownership',
                            'initial_list_status',
                            'verification_status',
                            'mths_since_earliest_cr_line',
                            'addr_state']]

df_validation_eng = apply_manual_bins(df_validation_eng)

df_train_eng = df_train_pd[['int_rate',
                            'sub_grade',
                            'grade',
                            'annual_inc',
                            'term_int',
                            'inq_last_6mths',
                            'total_rev_hi_lim',
                            'purpose',
                            'dti',
                            'home_ownership',
                            'initial_list_status',
                            'verification_status',
                            'mths_since_earliest_cr_line',
                            'addr_state']]

df_train_eng = apply_manual_bins(df_train_eng)


In [0]:
display(spark.createDataFrame(df_train_eng))

In [0]:
display(spark.createDataFrame(df_validation_eng))

In [0]:
best_features, results = select_features(
    df_train_eng, df_train_pd['default_bin'],
    df_validation_eng, df_validation_pd['default_bin'],
    all_features=features_selected_iv,
    reference_dict=reference_dict,
    alpha=0.05,
    many_cats_threshold=10,
    verbose=True
)

report_df = print_model_report(best_features, results, reference_dict)

In [0]:
best_features

In [0]:
display(spark.createDataFrame(report_df))

### Training and Evaluating our Final Model

In [0]:
final_features = ['int_rate',
                'sub_grade',
                'annual_inc',
                'inq_last_6mths',
                'total_rev_hi_lim',
                'purpose',
                'dti',
                'home_ownership',
                'initial_list_status',
                'verification_status',
                'mths_since_earliest_cr_line',
                'addr_state']

In [0]:
def feature_binning(df_in):

    df = df_in.copy()

    # ─────────────────────────────────────────────
    # NUMERICAL FEATURES
    # ─────────────────────────────────────────────

    # dti
    df['dti'] = pd.cut(df['dti'],
        bins=[-np.inf, 8.89, 10.30, 12.14, 13.38, 14.53, 15.59, 17.13, 18.80, 20.18, 21.44, 25.85, 29.01, np.inf],
        labels=['(-inf, 8.89)', '[8.89, 10.30)', '[10.30, 12.14)', '[12.14, 13.38)',
                '[13.38, 14.53)', '[14.53, 15.59)', '[15.59, 17.13)', '[17.13, 18.80)',
                '[18.80, 20.18)', '[20.18, 21.44)', '[21.44, 25.85)', '[25.85, 29.01)', '[29.01, inf)'])

    # annual_inc
    df['annual_inc'] = pd.cut(df['annual_inc'],
        bins=[-np.inf, 28338.00, 37086.20, 40371.00, 49464.10, 60996.50,
              66098.00, 70703.40, 80046.22, 90230.50, 100131.00, 120012.00, np.inf],
        labels=['(-inf, 28338.00)', '[28338.00, 37086.20)', '[37086.20, 40371.00)',
                '[40371.00, 49464.10)', '[49464.10, 60996.50)', '[60996.50, 66098.00)',
                '[66098.00, 70703.40)', '[70703.40, 80046.22)', '[80046.22, 90230.50)',
                '[90230.50, 100131.00)', '[100131.00, 120012.00)', '[120012.00, inf)'])

    # int_rate
    df['int_rate'] = pd.cut(df['int_rate'],
        bins=[-np.inf, 7.74, 8.92, 10.15, 11.01, 12.01, 13.05, 13.98, 15.12, 15.61, 17.57, 19.01, 20.99, np.inf],
        labels=['(-inf, 7.74)', '[7.74, 8.92)', '[8.92, 10.15)', '[10.15, 11.01)',
                '[11.01, 12.01)', '[12.01, 13.05)', '[13.05, 13.98)', '[13.98, 15.12)',
                '[15.12, 15.61)', '[15.61, 17.57)', '[17.57, 19.01)', '[19.01, 20.99)', '[20.99, inf)'])

    # total_rev_hi_lim
    df['total_rev_hi_lim'] = pd.cut(df['total_rev_hi_lim'],
        bins=[-np.inf, 5940.50, 11726.00, 20294.50, 28118.50, 36034.50, 44662.00, 55862.00, np.inf],
        labels=['(-inf, 5940.50)', '[5940.50, 11726.00)', '[11726.00, 20294.50)',
                '[20294.50, 28118.50)', '[28118.50, 36034.50)', '[36034.50, 44662.00)',
                '[44662.00, 55862.00)', '[55862.00, inf)'])

    # inq_last_6mths
    df['inq_last_6mths'] = pd.cut(df['inq_last_6mths'],
        bins=[-np.inf, 0.50, 1.50, 2.50, np.inf],
        labels=['(-inf, 0.50)', '[0.50, 1.50)', '[1.50, 2.50)', '[2.50, inf)'])

    # mths_since_earliest_cr_line
    df['mths_since_earliest_cr_line'] = pd.cut(df['mths_since_earliest_cr_line'],
        bins=[-np.inf, 145.50, 168.50, 204.50, 228.50, 247.50, 266.50, 287.50, 353.50, np.inf],
        labels=['(-inf, 145.50)', '[145.50, 168.50)', '[168.50, 204.50)', '[204.50, 228.50)',
                '[228.50, 247.50)', '[247.50, 266.50)', '[266.50, 287.50)', '[287.50, 353.50)', '[353.50, inf)'])

    # ─────────────────────────────────────────────
    # CATEGORICAL FEATURES
    # ─────────────────────────────────────────────

    # purpose
    purpose_map = {
        'credit_card':        "['credit_card', 'car']",
        'car':                "['credit_card', 'car']",
        'major_purchase':     "['major_purchase', 'home_improvement']",
        'home_improvement':   "['major_purchase', 'home_improvement']",
        'wedding':            "['wedding', 'debt_consolidation', 'vacation']",
        'debt_consolidation': "['wedding', 'debt_consolidation', 'vacation']",
        'vacation':           "['wedding', 'debt_consolidation', 'vacation']",
        'medical':            "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'house':              "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'other':              "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'moving':             "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'renewable_energy':   "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'small_business':     "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']",
        'educational':     "['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']"
    }
    df['purpose'] = df['purpose'].map(purpose_map).fillna("['medical', 'house', 'other', 'moving', 'renewable_energy', 'educational', 'small_business']") # fill NaN with the worst case scenario

    # initial_list_status
    df['initial_list_status'] = df['initial_list_status'].map({
        'w': "['w']",
        'f': "['f']",
    }).fillna("['f']") # fill NaN with the worst case scenario

    # verification_status
    df['verification_status'] = df['verification_status'].map({
        'Not Verified':    "['Not Verified']",
        'Source Verified': "['Source Verified']",
        'Verified':        "['Verified']",
    }).fillna("['Verified']") # fill NaN with the worst case scenario

    # addr_state
    addr_state_map = {
        # Bin 1
        'ME': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'DC': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'WY': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'ID': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'NH': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'WV': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'AK': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'CO': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'KS': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        'MS': "['ME', 'DC', 'WY', 'ID', 'NH', 'WV', 'AK', 'CO', 'KS', 'MS']",
        # Bin 2
        'MT': "['MT', 'VT', 'SC', 'TX']",
        'VT': "['MT', 'VT', 'SC', 'TX']",
        'SC': "['MT', 'VT', 'SC', 'TX']",
        'TX': "['MT', 'VT', 'SC', 'TX']",
        # Bin 3
        'CT': "['CT', 'IL']",
        'IL': "['CT', 'IL']",
        # Bin 4
        'OR': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'WI': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'WA': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'MN': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'GA': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        'SD': "['OR', 'WI', 'WA', 'MN', 'GA', 'SD']",
        # Bin 5
        'DE': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'MA': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'IN': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'KY': "['DE', 'MA', 'IN', 'KY', 'RI']",
        'RI': "['DE', 'MA', 'IN', 'KY', 'RI']",
        # Bin 6
        'OH': "['OH', 'PA', 'AZ', 'LA']",
        'PA': "['OH', 'PA', 'AZ', 'LA']",
        'AZ': "['OH', 'PA', 'AZ', 'LA']",
        'LA': "['OH', 'PA', 'AZ', 'LA']",
        # Bin 7
        'UT': "['UT', 'MI', 'VA']",
        'MI': "['UT', 'MI', 'VA']",
        'VA': "['UT', 'MI', 'VA']",
        # Bin 8
        'CA': "['CA', 'TN', 'AR']",
        'TN': "['CA', 'TN', 'AR']",
        'AR': "['CA', 'TN', 'AR']",
        # Bin 9
        'NC': "['NC', 'OK', 'MD']",
        'OK': "['NC', 'OK', 'MD']",
        'MD': "['NC', 'OK', 'MD']",
        # Bin 10
        'MO': "['MO', 'NY', 'NJ', 'NM']",
        'NY': "['MO', 'NY', 'NJ', 'NM']",
        'NJ': "['MO', 'NY', 'NJ', 'NM']",
        'NM': "['MO', 'NY', 'NJ', 'NM']",
        # Bin 11
        'AL': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'HI': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'FL': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'NV': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'IA': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
        'NE': "['AL', 'HI', 'FL', 'NV', 'IA', 'NE']",
    }
    df['addr_state'] = df['addr_state'].map(addr_state_map).fillna("['AL', 'HI', 'FL', 'NV', 'IA', 'NE']") # fill NaN with the worst case scenario

    # home_ownership
    df['home_ownership'] = df['home_ownership'].map({
        'MORTGAGE': "['MORTGAGE']",
        'OWN':      "['OWN']",
        'RENT':     "['RENT', 'NONE', 'OTHER']",
        'NONE':     "['RENT', 'NONE', 'OTHER']",
        'OTHER':    "['RENT', 'NONE', 'OTHER']",
    }).fillna("['RENT', 'NONE', 'OTHER']") # fill NaN with the worst case scenario

    # sub_grade
    last_bin = "['E2', 'E3', 'E4', 'F1', 'E5', 'F2', 'G4', 'F3', 'F4', 'G2', 'G5', 'G3', 'F5', 'G1']"
    sub_grade_map = {
        'A1': "['A1', 'A2', 'A3']", 'A2': "['A1', 'A2', 'A3']", 'A3': "['A1', 'A2', 'A3']",
        'A4': "['A4', 'A5']",       'A5': "['A4', 'A5']",
        'B1': "['B1']",
        'B2': "['B2']",
        'B3': "['B3']",
        'B4': "['B4']",
        'B5': "['B5']",
        'C1': "['C1']",
        'C2': "['C2']",
        'C3': "['C3']",
        'C4': "['C4']",
        'C5': "['C5', 'D1']",       'D1': "['C5', 'D1']",
        'D2': "['D2', 'D3']",       'D3': "['D2', 'D3']",
        'D4': "['D4', 'D5', 'E1']", 'D5': "['D4', 'D5', 'E1']", 'E1': "['D4', 'D5', 'E1']",
        'E2': last_bin, 'E3': last_bin, 'E4': last_bin, 'E5': last_bin,
        'F1': last_bin, 'F2': last_bin, 'F3': last_bin, 'F4': last_bin, 'F5': last_bin,
        'G1': last_bin, 'G2': last_bin, 'G3': last_bin, 'G4': last_bin, 'G5': last_bin,
    }
    df['sub_grade'] = df['sub_grade'].map(sub_grade_map).fillna("['E2', 'E3', 'E4', 'F1', 'E5', 'F2', 'G4', 'F3', 'F4', 'G2', 'G5', 'G3', 'F5', 'G1']") # fill NaN with the worst case scenario

    return df

In [0]:
target = 'default_bin'

df_validation_pd = df_validation.toPandas()

df_validation_eng = df_validation_pd[final_features + [target]]

df_validation_eng = feature_binning(df_validation_eng)

df_train_eng = df_train_pd[final_features + [target]]

df_train_eng = feature_binning(df_train_eng)


In [0]:
class LogisticRegressionWithPValues(BaseEstimator, ClassifierMixin):
    def __init__(self, max_iter=1000):
        self.max_iter = max_iter

    def fit(self, X, y):
        self.model_ = LogisticRegression(C=1e9, max_iter=self.max_iter, solver='lbfgs')
        self.model_.fit(X, y)
        denom = 2.0 * (1.0 + np.cosh(self.model_.decision_function(X)))
        denom = np.tile(denom, (X.shape[1], 1)).T
        F_ij = np.dot((X / denom).T, X)
        Cramer_Rao = np.linalg.inv(F_ij)
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model_.coef_[0] / sigma_estimates
        self.p_values_ = [stat.norm.sf(abs(x)) * 2 for x in z_scores]
        self.coef_ = self.model_.coef_
        self.intercept_ = self.model_.intercept_
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        check_is_fitted(self)
        return self.model_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self)
        return self.model_.predict_proba(X)


class CalibratedPDModel(mlflow.pyfunc.PythonModel):
    """
    Wraps the base pipeline + Platt scaler into a single MLflow model.
    Input  : DataFrame with raw (string-typed) binned feature columns
    Output : DataFrame with a single column 'pd_calibrated'
    """

    def __init__(self, pipeline, platt_scaler, features):
        self.pipeline     = pipeline
        self.platt_scaler = platt_scaler
        self.features     = features

    def predict(self, context, model_input):
        X_raw     = model_input[self.features].astype(str)
        raw_proba = self.pipeline.predict_proba(X_raw)[:, 1]
        cal_proba = self.platt_scaler.predict_proba(
            raw_proba.reshape(-1, 1)
        )[:, 1]
        return cal_proba


# ─────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────

def compute_ks(y_true, y_proba):
    bads  = y_proba[y_true == 1]
    goods = y_proba[y_true == 0]
    ks_stat, _ = ks_2samp(bads, goods)
    return round(ks_stat, 4)


def compute_roc_auc(y_true, y_proba):
    return round(roc_auc_score(y_true, y_proba), 4)


# ─────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────

def plot_roc_curve(y_true, y_proba, title='ROC Curve — Validation'):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right')
    plt.tight_layout()
    return fig


def plot_ks_curve(y_true, y_proba, title='KS Curve — Validation'):
    df_plot = pd.DataFrame({'score': y_proba, 'target': y_true})
    df_plot = df_plot.sort_values('score', ascending=False).reset_index(drop=True)
    n = len(df_plot)
    df_plot['cum_bad']  = (df_plot['target'] == 1).cumsum() / (y_true == 1).sum()
    df_plot['cum_good'] = (df_plot['target'] == 0).cumsum() / (y_true == 0).sum()
    df_plot['ks']       = df_plot['cum_bad'] - df_plot['cum_good']
    ks_idx  = df_plot['ks'].abs().idxmax()
    ks_stat = df_plot['ks'].abs().max()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(np.arange(n) / n, df_plot['cum_bad'],  color='firebrick', lw=2, label='Cumulative Bad')
    ax.plot(np.arange(n) / n, df_plot['cum_good'], color='steelblue', lw=2, label='Cumulative Good')
    ax.axvline(x=ks_idx / n, color='gray', linestyle='--', lw=1)
    ax.annotate(f'KS = {ks_stat:.4f}',
                xy=(ks_idx / n, (df_plot['cum_bad'][ks_idx] + df_plot['cum_good'][ks_idx]) / 2),
                fontsize=10, color='gray')
    ax.set_xlabel('Population (sorted by score descending)')
    ax.set_ylabel('Cumulative Rate')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    return fig


# def plot_calibration_curve(y_true, y_proba, n_bins=10, title='Calibration Curve — Validation'):
#     prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=n_bins, strategy='uniform')
    
#     bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
#     bin_sizes_all = np.histogram(y_proba, bins=bin_edges)[0]

#     non_empty = bin_sizes_all > 0
#     bin_sizes = bin_sizes_all[non_empty]

#     ece = np.sum(np.abs(prob_true - prob_pred) * bin_sizes) / len(y_true)
#     fig, ax = plt.subplots(figsize=(7, 5))
#     ax.plot(prob_pred, prob_true, 's-', color='steelblue', lw=2, label=f'Model (ECE={ece:.4f})')
#     ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
#     ax.set_xlabel('Mean Predicted Probability')
#     ax.set_ylabel('Fraction of Positives')
#     ax.set_title(title)
#     ax.legend()
#     plt.tight_layout()
#     return fig, ece

def plot_calibration_curve(
    y_true,
    y_proba_raw,
    y_proba_cal,
    n_bins=10,
    title='Calibration Curve — Validation'
):
    """
    Plots raw and Platt-calibrated calibration curves on the same graph.

    Parameters
    ----------
    y_true      : array-like, true binary labels
    y_proba_raw : array-like, uncalibrated predicted probabilities
    y_proba_cal : array-like, Platt-calibrated predicted probabilities
    n_bins      : int, number of bins for calibration curve
    title       : str, plot title

    Returns
    -------
    fig         : matplotlib Figure
    ece_raw     : float, ECE before calibration
    ece_cal     : float, ECE after calibration
    """

    def _compute_curve_and_ece(y_true, y_proba):
        prob_true, prob_pred = calibration_curve(
            y_true, y_proba, n_bins=n_bins, strategy='uniform'
        )
        bin_edges     = np.linspace(0.0, 1.0, n_bins + 1)
        bin_sizes_all = np.histogram(y_proba, bins=bin_edges)[0]
        non_empty     = bin_sizes_all > 0
        bin_sizes     = bin_sizes_all[non_empty]
        ece = np.sum(np.abs(prob_true - prob_pred) * bin_sizes) / len(y_true)
        return prob_true, prob_pred, ece

    prob_true_raw, prob_pred_raw, ece_raw = _compute_curve_and_ece(y_true, y_proba_raw)
    prob_true_cal, prob_pred_cal, ece_cal = _compute_curve_and_ece(y_true, y_proba_cal)

    fig, ax = plt.subplots(figsize=(8, 6))

    # Perfect calibration reference
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')

    # Raw curve
    ax.plot(
        prob_pred_raw, prob_true_raw,
        's-', color='tomato', lw=2,
        label=f'Raw (ECE={ece_raw:.4f})'
    )

    # Calibrated curve
    ax.plot(
        prob_pred_cal, prob_true_cal,
        's-', color='steelblue', lw=2,
        label=f'Platt calibrated (ECE={ece_cal:.4f})'
    )

    # ECE improvement annotation
    delta = ece_raw - ece_cal
    ax.annotate(
        f'ECE improvement: {delta:+.4f}',
        xy=(0.98, 0.04),
        xycoords='axes fraction',
        ha='right',
        fontsize=9,
        color='dimgray',
        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray', alpha=0.8)
    )

    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()

    return fig, ece_raw, ece_cal


# ─────────────────────────────────────────────
# COEFFICIENT TABLE
# ─────────────────────────────────────────────

def build_coefficient_table(pipeline, final_features):
    encoder = pipeline.named_steps['encoder']
    model   = pipeline.named_steps['model']
    feature_names = encoder.get_feature_names_out(final_features)
    coef_df = pd.DataFrame({
        'Feature_Category': feature_names,
        'Coefficient':      model.coef_[0],
        'P_Value':          model.p_values_,
    })
    coef_df['Feature']     = coef_df['Feature_Category'].apply(lambda x: '_'.join(x.split('_')[:-1]))
    coef_df['Category']    = coef_df['Feature_Category'].apply(lambda x: x.split('_')[-1])
    coef_df['Significant'] = coef_df['P_Value'] < 0.05

    return coef_df[['Feature', 'Category', 'Coefficient', 'P_Value', 'Significant']] \
                  .sort_values(['Feature', 'Coefficient'], ascending=[True, False]) \
                  .reset_index(drop=True)


In [0]:
# ─────────────────────────────────────────────
# MAIN TRAINING PIPELINE
# ─────────────────────────────────────────────

def train_pd_model(
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    final_features: list,
    target: str,
    reference_dict: dict,
    experiment_name: str = '/Shared/pd_scorecard',
):
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run():

        # ── 1. Train / Validation split ───────────────────────────────
        X_train_raw = df_train[final_features].astype(str)
        y_train = df_train[target]
        
        X_val_val, X_val_test, y_val_val, y_val_test = train_test_split(df_val[final_features].astype(str), df_val[target], test_size=0.4, random_state=42, stratify=df_val[target])

        # ── 2. Validate reference categories ─────────────────────────
        enc_temp = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        enc_temp.fit(X_train_raw)

        drop_cats = []
        for feat, cats in zip(final_features, enc_temp.categories_):
            ref = reference_dict[feat]
            if ref not in cats:
                raise ValueError(
                    f"\nReference mismatch for '{feat}'."
                    f"\n  Expected : '{ref}'"
                    f"\n  Available: {list(cats)}"
                )
            drop_cats.append(ref)

        # ── 3. Build and fit sklearn pipeline ─────────────────────────
        pipeline = Pipeline(steps=[
            ('encoder', OneHotEncoder(
                drop=drop_cats,
                sparse_output=False,
                handle_unknown='ignore'
            )),
            ('model', LogisticRegressionWithPValues(max_iter=1000))
        ])

        pipeline.fit(X_train_raw, y_train)


        # ── 4. Predict ────────────────────────────────────────────────
        y_val_val_proba_raw = pipeline.predict_proba(X_val_val)[:, 1]
        y_val_test_proba_raw   = pipeline.predict_proba(X_val_test)[:, 1]

        y_train_proba_raw   = pipeline.predict_proba(X_train_raw)[:, 1]

        # Fit Platt scaler on validation scores — never on training,
        # as the base model already saw that data
        from sklearn.linear_model import LogisticRegression as _LR
        platt_scaler = _LR(C=1e9, solver='lbfgs')
        platt_scaler.fit(y_val_val_proba_raw.reshape(-1, 1), y_val_val)

        # Calibrated probabilities
        y_val_test_proba_cal   = platt_scaler.predict_proba(
            y_val_test_proba_raw.reshape(-1, 1)
        )[:, 1]

        # ── 5. Metrics ────────────────────────────────────────────────
        train_auc = compute_roc_auc(y_train, y_train_proba_raw)
        val_auc   = compute_roc_auc(y_val_test,   y_val_test_proba_cal)
        train_ks  = compute_ks(y_train, y_train_proba_raw)
        val_ks    = compute_ks(y_val_test,   y_val_test_proba_cal)

        mlflow.log_metrics({
            'train_roc_auc':    train_auc,
            'val_roc_auc':      val_auc,
            'train_ks':         train_ks,
            'val_ks':           val_ks
        })

        print(f"Train AUC: {train_auc} | Val AUC: {val_auc}")
        print(f"Train KS:  {train_ks}  | Val KS:  {val_ks}")

        # ── 6. Log parameters ─────────────────────────────────────────
        mlflow.log_params({
            'features':       str(final_features),
            'n_features':     len(final_features),
            'target':         target,
            'regularization': 'none (C=1e9)',
        })

        # ── 7. Coefficient table ──────────────────────────────────
        coef_df = build_coefficient_table(pipeline, final_features)
        mlflow.log_text(coef_df.to_csv(index=False), 'model_diagnostics/coefficients.csv')

        # ── 8. ROC plot ───────────────────────────────────────────
        fig_roc = plot_roc_curve(y_val_test, y_val_test_proba_cal)
        mlflow.log_figure(fig_roc, 'model_diagnostics/roc_curve_validation.png')
        plt.close(fig_roc)

        # ── 9. KS plot ────────────────────────────────────────────
        fig_ks = plot_ks_curve(y_val_test, y_val_test_proba_cal)
        mlflow.log_figure(fig_ks, 'model_diagnostics/ks_curve_validation.png')
        plt.close(fig_ks)

        # ── 10. Calibration plot ──────────────────────────────────
        fig_cal, ece_raw, ece_cal = plot_calibration_curve(y_val_test, y_val_test_proba_raw, y_val_test_proba_cal)
        mlflow.log_figure(fig_cal, 'model_diagnostics/calibration_curve_validation.png')
        plt.close(fig_cal)
        mlflow.log_metric('val_ece_raw', round(ece_raw, 6))
        mlflow.log_metric('val_ece_cal', round(ece_cal, 6))

        # ── 11. Log sklearn pipeline ──────────────────────────────

        calibrated_model = CalibratedPDModel(
            pipeline     = pipeline,
            platt_scaler = platt_scaler,
            features     = final_features,
        )

        mlflow.pyfunc.log_model(
            artifact_path    = 'pd_model',
            python_model     = calibrated_model,
            pip_requirements = [
                'scikit-learn==1.4.2',
                'pandas',
                'numpy',
                'scipy',
            ]
        )

        run_id = mlflow.active_run().info.run_id
        print(f"[OK] Run logged. Run ID: {run_id}")

    return run_id, coef_df

In [0]:
# ─────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────

run_id, coef_df = train_pd_model(
    df_train=df_train_eng,
    df_val=df_validation_eng,
    final_features=final_features,
    target=TARGET,
    reference_dict=reference_dict,
    experiment_name='/Shared/pd_scorecard_v1',
)


### Using the model - Testing on unseen data

In [0]:
df_test_pd = df_test.toPandas()

df_test_eng = df_test_pd[final_features]

df_test_eng = feature_binning(df_test_eng)

In [0]:
# run_id = '84d7678accf04c9b8e1ad3cb93c598ff'

y_test = df_test_pd['default_bin'].values

loaded_model = mlflow.pyfunc.load_model(f'runs:/{run_id}/pd_model')
pd_scores = loaded_model.predict(df_test_eng.astype(str))


# loaded_pipeline = mlflow.sklearn.load_model(f'runs:/{run_id}/pd_model')
# pd_scores = loaded_pipeline.predict_proba(df_test_eng.astype(str))[:, 1]

fig_roc_test = plot_roc_curve(y_test, pd_scores)

fig_ks_test = plot_ks_curve(y_test, pd_scores)